# BEAR Xbox Teleop — 3-Motor Test

Drives up to three **Koala BEAR Muscle Build (KBMB02)** actuators (IDs 1, 2, 3 on one RS-485
bus/dangle) with an Xbox controller. **All three run in Velocity mode** — speed is proportional to
how far each stick/trigger is deflected ("depth of input"), all the way back to zero at rest.

**Control mapping:**

| Control | Role | Mode | Direction |
|---|---|---|---|
| Left stick (L3), up/down | `left_stick_speed` | Velocity, high torque ceiling | up = CW, down = CCW |
| Right stick (R3), left/right | `right_stick_speed` | Velocity, high torque ceiling | right = CW, left = CCW |
| LT / RT triggers | `trigger_speed` | Velocity | RT = CW, LT = CCW |

**Motors can be tested independently.** Section 1's `ROLE_TO_ID` is the single on/off switch —
comment out any role whose motor isn't currently connected, and every downstream cell (ping,
configure, control loop) only touches what's left active. Physical motor **IDs are mapped to
roles**, not hardcoded to axes, so if you rewire which motor sits in which mechanical slot, or
swap two motors' IDs, edit that one dict and nothing else needs to change.

**Depth of input maps directly to speed.** Each control's normalized deflection (−1..1 for sticks,
0..1 combined into −1..1 for the trigger pair) is multiplied by that role's `limit_velocity_max` and
sent straight to `goal_velocity` every tick — light touch = slow, full deflection = max configured
speed. No jogging or position tracking involved.

**"Max torque" means the torque *ceiling* is raised, not that goal_iq is slammed to max.**
`limit_i_max` still runs through PyBEAR's normal PID cascade; raising it just lets the velocity
loop use more current when it needs to. KBMB02 torque constant is 1.16 Nm/A with 6.0 Nm continuous
*Rated Torque* (≈5.2 A) and short-pulse *Stall Torque* up to 20 Nm (≈17 A) per the SDK manual (Table 1).
Defaults below start at roughly half of rated current — strong, but with headroom to verify
direction/behavior before pushing toward the rated/stall numbers.

**Controls drive their motors directly** — no extra button needs to be held. **B** is a hard kill,
disabling all three motors immediately; `Ctrl-C` / the notebook stop button does the same cleanup
via a `finally` block.

**Safety built in:**
- All torque/speed/deadzone values are in one config cell (Section 1) — tune freely.
- `limit_velocity_max` per role caps the top speed regardless of how hard a control is deflected.
- Run the **EMERGENCY DISABLE** cell any time; it's idempotent and safe to spam. Unlike everything
  else in this notebook, EMERGENCY DISABLE and Cleanup always target **all three possible IDs**,
  not just whatever's currently active in `ROLE_TO_ID` — so if you disable a motor's role after
  already enabling it, this still reaches it.


## 1. Active motors & tuning

**Comment out (or delete) any line in `ROLE_TO_ID` for a motor that isn't currently connected.**
Only the roles left here get pinged, configured, enabled, and driven by the rest of the notebook —
this is what lets you test one or two motors independently while others are unplugged.

Torque/speed/deadzone values below are a starting middle ground — raise or lower after testing.

In [157]:
import math

# --- Which motor ID plays which control role. All roles run in Velocity mode. ---
# Comment out (or delete) any line below for a motor that ISN'T currently connected — only the
# roles left active here get pinged/configured/enabled/driven by the rest of the notebook.
# Swap ID numbers here too if motors get rewired/renumbered.
ROLE_TO_ID = {
    'left_stick_speed':  3,   # Velocity mode, driven by LEFT stick (L3) depth: up = CW, down = CCW
    'right_stick_speed': 1,   # Velocity mode, driven by RIGHT stick (R3) depth: right = CW, left = CCW
    'trigger_speed':     2,   # Velocity mode, driven by triggers: RT = CW, LT = CCW
}

# --- Per-role tuning. Torque/speed values are a starting middle ground — raise or lower after testing. ---
ROLE_PARAMS = {
    'left_stick_speed': dict(
        limit_i_max         = 3.0,   # A. KBMB02 rated current ~5.2 A (6.0 Nm / 1.16 Nm/A); this is ~mid-way there.
        limit_velocity_max   = 4.0,  # rad/s commanded at full stick deflection.
        limit_position_min  = -16 * math.pi,  # rad. SDK max range is -8pi..8pi; widened from a narrower prior setting.
        limit_position_max  = 16 * math.pi,   # rad.
        invert                = False,  # flip to True if "up" spins the wrong physical direction.
    ),
    'right_stick_speed': dict(
        limit_i_max         = 3.0,
        limit_velocity_max   = 4.0,
        limit_position_min  = -16 * math.pi,
        limit_position_max  = 16 * math.pi,
        invert                = False,  # flip to True if "right" spins the wrong physical direction.
    ),
    'trigger_speed': dict(
        limit_i_max         = 2.0,   # A. Moderate default for continuous velocity-mode driving.
        limit_velocity_max   = 6.0,  # rad/s commanded at full trigger pull.
        limit_position_min  = -16 * math.pi,
        limit_position_max  = 16 * math.pi,
        invert                = False,  # flip to True if RT spins the wrong physical direction.
    ),
}

# --- Global control-loop settings ---
CONTROL_HZ        = 50            # control loop update rate
STATUS_INTERVAL_S = 0.5           # how often to print the status line
STICK_DEADZONE    = 0.12          # fraction of full deflection ignored near center
TRIGGER_DEADZONE  = 0.05
HEALTH_CHECK_EVERY_N_TICKS = 25   # ~2x/sec at 50 Hz: re-check torque_enable / error bits

print("Active roles -> ID:", ROLE_TO_ID)


Active roles -> ID: {'left_stick_speed': 3, 'right_stick_speed': 1, 'trigger_speed': 2}


## 2. Connect to the BEAR bus

All motors share one RS-485 dangle/port — one `Manager.BEAR` instance, addressed per-packet by ID.
Only pings the IDs currently active in `ROLE_TO_ID` above.

In [158]:
import time
from pybear import Manager

PORT = '/dev/serial/by-id/usb-Westwood_Robotics_USB2BEAR_Boosted_UB0104-if00-port0'
BAUD = 8000000

bear = Manager.BEAR(
    port=PORT,
    baudrate=BAUD,
    single_timeout=0.01,
    single_try_num=3,
    bulk_timeout=0.01,
    bulk_try_num=3,
)

active_ids = sorted(set(ROLE_TO_ID.values()))
missing = []
for mid in active_ids:
    _, err = bear.ping(mid)[0]
    if err is None:
        missing.append(mid)
    else:
        print(f"BEAR {mid} online (error byte 0x{err:02x}: {bear.decode_error(err)})")

if missing:
    print(f"\nWARNING: no response from ID(s) {missing}, but they're active in ROLE_TO_ID (Section 1). "
          f"Either fix the connection, or comment that role out in Section 1 and re-run from there.")


   __        __          _                               _    
   \ \      / /___  ___ | |_ __      __ ___    ___    __| |   
    \ \ /\ / // _ \/ __|| __|\ \ /\ / // _ \  / _ \  / _` |   
     \ V  V /|  __/\__ \| |_  \ V  V /| (_) || (_) || (_| |   
      \_/\_/  \___||___/ \__|  \_/\_/  \___/  \___/  \__,_|   
              ____         _             _    _               
             |  _ \  ___  | |__    ___  | |_ (_)  ___  ___    
             | |_) |/ _ \ | '_ \  / _ \ | __|| | / __|/ __|   
             |  _ <| (_) || |_) || (_) || |_ | || (__ \__ \   
             |_| \_\___/ |_.__/  \___/  \__||_| \___||___/   
                                                              
=== PyBEAR by Westwood Robotics -- Last Updated 2025.07.29 ===

BEAR 1 online (error byte 0x80: No Error!)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
BEAR 2 online (error byte 0x80: No Error!)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
BEAR 3 online (error byte 0x80: 

## 3. Xbox controller

Uses `evdev` (Linux native, no display/SDL needed). Installs it if missing, then autodetects the controller by name.

In [159]:
try:
    import evdev
    from evdev import ecodes
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'evdev'], check=True)
    import evdev
    from evdev import ecodes

def find_controller():
    candidates = []
    for path in evdev.list_devices():
        dev = evdev.InputDevice(path)
        if any(k in dev.name for k in ('Xbox', 'X-Box', 'Microsoft')):
            candidates.append(dev)
        else:
            dev.close()
    return candidates

candidates = find_controller()

if not candidates:
    print("No Xbox controller auto-detected. All input devices currently visible:")
    for path in evdev.list_devices():
        d = evdev.InputDevice(path)
        print(f"  {d.path}  {d.name!r}")
        d.close()
    raise RuntimeError(
        "Plug the controller in (wired, or via the official wireless adapter) and re-run this cell.\n"
        "If it's plugged in but not listed above, check permissions: `groups` should include `input` "
        "(run `sudo usermod -aG input $USER`, then log out/in) — otherwise /dev/input/eventN is unreadable."
    )

joy = candidates[0]
print(f"Using controller: {joy.name}  ({joy.path})")

caps = joy.capabilities(verbose=False)
abs_codes = {c for c, _ in caps.get(ecodes.EV_ABS, [])}
btn_codes = set(caps.get(ecodes.EV_KEY, []))
print(f"ABS axes present : {sorted(ecodes.ABS[c] if isinstance(ecodes.ABS[c], str) else ecodes.ABS[c][0] for c in abs_codes)}")
print(f"Buttons present  : {len(btn_codes)} buttons")

REQUIRED_ABS = {ecodes.ABS_Y, ecodes.ABS_RX, ecodes.ABS_Z, ecodes.ABS_RZ}
missing_abs = REQUIRED_ABS - abs_codes
if missing_abs:
    print(f"\nWARNING: expected axes missing: {[ecodes.ABS[c] for c in missing_abs]}. "
          f"Your controller/driver may map sticks or triggers to different ABS codes — "
          f"check the 'ABS axes present' list above and adjust the AXIS_* constants in Section 5.")

# --- Live check: confirms which ABS code LT/RT actually fire on THIS controller/driver. ---
# This notebook assumes ABS_Z = LT and ABS_RZ = RT (the standard Linux xpad mapping for Xbox
# controllers), but Bluetooth stacks / older drivers occasionally differ. If LT/RT don't seem to
# do anything later, the codes printed below (not the assumption above) are the ground truth —
# update AXIS_LT/AXIS_RT in Section 5 to match if they differ from ABS_Z/ABS_RZ.
import select
print("\nPull LT then RT now — sampling for 3 seconds...")
seen = {}
t_end = time.time() + 3.0
while time.time() < t_end:
    if select.select([joy.fd], [], [], 0.05)[0]:
        for event in joy.read():
            if event.type == ecodes.EV_ABS:
                seen[event.code] = event.value

if not seen:
    print("  No ABS events seen. Try again and make sure to actually move a stick/trigger during the 3s window.")
else:
    for code, val in sorted(seen.items()):
        name = ecodes.ABS[code] if isinstance(ecodes.ABS[code], str) else ecodes.ABS[code][0]
        flag = "  <-- expected LT" if code == ecodes.ABS_Z else "  <-- expected RT" if code == ecodes.ABS_RZ else ""
        print(f"  code {code:3d} ({name}): last value = {val}{flag}")
    if ecodes.ABS_Z not in seen or ecodes.ABS_RZ not in seen:
        print("  ABS_Z and/or ABS_RZ never changed — LT/RT are firing under a different code on this "
              "controller. Use the code(s) printed above to update AXIS_LT/AXIS_RT in Section 5.")


Using controller: Microsoft Xbox Series S|X Controller  (/dev/input/event18)
ABS axes present : ['ABS_HAT0X', 'ABS_HAT0Y', 'ABS_RX', 'ABS_RY', 'ABS_RZ', 'ABS_X', 'ABS_Y', 'ABS_Z']
Buttons present  : 12 buttons

Pull LT then RT now — sampling for 3 seconds...
  No ABS events seen. Try again and make sure to actually move a stick/trigger during the 3s window.


### 3b. Manual LT/RT isolation test (run if the automatic check above was inconclusive)

The 3-second free-form test can't tell a deliberate trigger pull apart from an incidental stick
bump. This one prompts you through each control **one at a time** and reads the axis's true
instantaneous value directly (not just whatever happened to generate an event), so there's no
ambiguity about which code moved. Follow the prompts exactly — don't touch anything except what's
asked at each step.

In [ ]:
def _read_all_abs():
    """Direct EVIOCGABS read of every ABS axis's current value — not event-dependent."""
    return {code: joy.absinfo(code).value for code, _ in caps.get(ecodes.EV_ABS, [])}

def _name(code):
    return ecodes.ABS[code] if isinstance(ecodes.ABS[code], str) else ecodes.ABS[code][0]

def _diff(before, after):
    return {c: (before[c], after[c]) for c in before if after[c] != before[c]}

def _report(label, changed):
    if not changed:
        print(f"  Nothing changed while holding {label}. Are you sure it's pulled all the way, and is this the right controller?")
    else:
        for c, (b, a) in changed.items():
            print(f"  {label} -> code {c} ({_name(c)}): {b} -> {a}")

input("Release ALL sticks and triggers completely, then press Enter...")
baseline = _read_all_abs()

input("Now pull and HOLD ONLY the LEFT trigger (LT) — don't touch anything else — then press Enter...")
lt_changed = _diff(baseline, _read_all_abs())
print("Left trigger (LT) result:")
_report("LT", lt_changed)

input("\nRelease LT. Now pull and HOLD ONLY the RIGHT trigger (RT) — don't touch anything else — then press Enter...")
rt_changed = _diff(baseline, _read_all_abs())
print("Right trigger (RT) result:")
_report("RT", rt_changed)

print("\nIn Section 5, set AXIS_LT / AXIS_RT to ecodes.<name printed above> if it differs from ABS_Z / ABS_RZ.")


## 4. Configure & enable each motor in its mode

Standard safe sequence per motor: **disable → set gains/limits → prime goal → set mode → enable**.
Every active motor gets the same treatment — velocity+id/iq gains, then Velocity mode. Uses the
same recommended id/iq gains as the single-motor test notebook. Only iterates over roles active in
`ROLE_TO_ID` (Section 1).

In [163]:
def read1(getter, motor_id):
    data_list, err = getter(motor_id)[0]
    return data_list[0], err

def disable(motor_id, tries=5, delay=0.01):
    for attempt in range(tries):
        bear.set_torque_enable((motor_id, 0))
        state, err = read1(bear.get_torque_enable, motor_id)
        if err is not None and state == 0:
            return
        time.sleep(delay)
    raise RuntimeError(f"BEAR {motor_id}: disable never confirmed after {tries} tries — check the link.")

def set_mode_confirmed(motor_id, new_mode, tries=5, delay=0.01):
    for attempt in range(tries):
        bear.set_mode((motor_id, new_mode))
        actual, err = read1(bear.get_mode, motor_id)
        if err is not None and actual == new_mode:
            return
        time.sleep(delay)
    raise RuntimeError(f"BEAR {motor_id}: set_mode({new_mode}) never confirmed after {tries} tries.")

MODE_VELOCITY = 1

for role, motor_id in ROLE_TO_ID.items():
    p = ROLE_PARAMS[role]
    disable(motor_id)

    # Current-loop gains (KBMB02 recommended, same as single-motor test notebook)
    bear.set_p_gain_id((motor_id, 0.358)); bear.set_i_gain_id((motor_id, 0.045)); bear.set_d_gain_id((motor_id, 0.0))
    bear.set_p_gain_iq((motor_id, 0.358)); bear.set_i_gain_iq((motor_id, 0.045)); bear.set_d_gain_iq((motor_id, 0.0))

    bear.set_limit_i_max((motor_id, p['limit_i_max']))
    bear.set_limit_velocity_max((motor_id, p['limit_velocity_max']))
    bear.set_limit_position_min((motor_id, p['limit_position_min']))
    bear.set_limit_position_max((motor_id, p['limit_position_max']))

    bear.set_p_gain_velocity((motor_id, 0.5)); bear.set_i_gain_velocity((motor_id, 0.0)); bear.set_d_gain_velocity((motor_id, 0.0))
    bear.set_goal_velocity((motor_id, 0.0))
    set_mode_confirmed(motor_id, MODE_VELOCITY)
    bear.set_torque_enable((motor_id, 1))
    print(f"[{role}] BEAR {motor_id}: Velocity mode enabled, limit_i_max={p['limit_i_max']} A, "
          f"limit_velocity_max={p['limit_velocity_max']} rad/s, "
          f"limit_position=[{p['limit_position_min']:.2f}, {p['limit_position_max']:.2f}] rad.")
    
print("\nAll active motors enabled. Proceed to the control loop (Section 6).")


[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[left_stick_speed] BEAR 3: Velocity mode enabled, limit_i_max=3.0 A, limit_velocity_max=4.0 rad/s, limit_position=[-50.27, 50.27] rad.
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[right_stick_speed] BEAR 1: Velocity mode enabled, limit_i_max=3.0 A, limit_velocity_max=4.0 rad/s, limit_position=[-50.27, 50.27] rad.
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[trigger_speed] BEAR 2: Velocity mode enabled, limit_i_max=2.0 A, limit_velocity_max=6.0 rad/s, limit_position=[-50.27, 50.27] rad.

All active motors enabled. Proceed to the control loop (Section 6).


## 5. Controller state reader

Drains all pending evdev events non-blocking each poll and keeps the latest raw value per axis/button.
Axis values are normalized using the device's own reported min/max, so this works across controller
variants without hardcoded ranges.

In [164]:
AXIS_LEFT_Y   = ecodes.ABS_Y     # left stick vertical  (L3)
AXIS_RIGHT_X  = ecodes.ABS_RX    # right stick horizontal (R3)
AXIS_LT       = ecodes.ABS_Z     # left trigger (LT)
AXIS_RT       = ecodes.ABS_RZ    # right trigger (RT)
BTN_KILL      = ecodes.BTN_EAST  # B  — immediate full stop
print(read1(bear.get_present_position, 3))
print(read1(bear.get_limit_position_min, 3))
print(read1(bear.get_limit_position_max, 3))
class ControllerState:
    def __init__(self, dev):
        self.dev = dev
        self._absinfo = {c: dev.absinfo(c) for c in (AXIS_LEFT_Y, AXIS_RIGHT_X, AXIS_LT, AXIS_RT)}
        self.raw = {c: 0 for c in self._absinfo}
        self.buttons = set()

    def poll(self):
        """Drain all pending events without blocking. Call once per control-loop tick."""
        while True:
            try:
                event = self.dev.read_one()
            except BlockingIOError:
                break
            if event is None:
                break
            if event.type == ecodes.EV_ABS and event.code in self.raw:
                self.raw[event.code] = event.value
            elif event.type == ecodes.EV_KEY:
                if event.value:
                    self.buttons.add(event.code)
                else:
                    self.buttons.discard(event.code)

    def stick(self, code, invert=False):
        """Normalized [-1, 1], deadzone-applied. Positive = up (for Y axes) / right (for X axes)."""
        info = self._absinfo[code]
        center = (info.max + info.min) / 2
        span = (info.max - info.min) / 2
        v = (self.raw[code] - center) / span
        v = -v if code == AXIS_LEFT_Y else v   # raw ABS_Y increases downward; flip so up = +1
        if invert:
            v = -v
        return _deadzone(v, STICK_DEADZONE)

    def trigger(self, code, invert=False):
        """Normalized [0, 1], deadzone-applied."""
        info = self._absinfo[code]
        v = (self.raw[code] - info.min) / (info.max - info.min)
        if invert:
            v = 1.0 - v
        return _deadzone(v, TRIGGER_DEADZONE, floor_zero=True)

def _deadzone(v, dz, floor_zero=False):
    lo = 0.0 if floor_zero else -1.0
    if abs(v) < dz:
        return 0.0
    span = 1.0 - dz
    return math.copysign((abs(v) - dz) / span, v)

ctrl = ControllerState(joy)   # evdev opens the device fd non-blocking by default
print("Controller state reader ready.")


(-2.986506223678589, 128)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
(-50.26548385620117, 128)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
(50.26548385620117, 128)
Controller state reader ready.


## 6. Live control loop

Every active role is Velocity mode: each control's deflection depth is scaled by that role's
`limit_velocity_max` and sent straight to `goal_velocity` — no extra button required, speed tracks
input in real time. Press **B** for an immediate full stop (disables all three motors). `Ctrl-C` /
the notebook's stop button does the same cleanup via `finally`.

Only drives roles active in `ROLE_TO_ID` (Section 1). Re-run Section 4 after a kill/interrupt to
re-enable before driving again.

In [165]:
dt = 1.0 / CONTROL_HZ
last_status = 0.0
tick = 0
last_warned = {}   # role -> last torque_enable value we already printed a warning for

print("Control loop running. Press B to kill, Ctrl-C / stop button to exit safely.\n")

try:
    while True:
        loop_t0 = time.time()
        ctrl.poll()

        if BTN_KILL in ctrl.buttons:
            print("\nB pressed — killing all motors.")
            break

        # Periodic health check: don't fight a latched error by continuing to spam goals at it.
        if tick % HEALTH_CHECK_EVERY_N_TICKS == 0:
            for role, motor_id in ROLE_TO_ID.items():
                state, _ = read1(bear.get_torque_enable, motor_id)
                if state != 1 and last_warned.get(role) != state:
                    print(f"\n[{role}] BEAR {motor_id} torque_enable={state} (not normally-enabled) — "
                          f"holding goals, not sending new commands until you re-run Section 4.")
                last_warned[role] = state

        for role, motor_id in ROLE_TO_ID.items():
            if last_warned.get(role, 1) != 1:
                continue   # this motor isn't healthy; leave it alone
            p = ROLE_PARAMS[role]

            if role == 'trigger_speed':
                lt = ctrl.trigger(AXIS_LT)
                rt = ctrl.trigger(AXIS_RT)
                cmd = (rt - lt)
            elif role == 'left_stick_speed':
                cmd = ctrl.stick(AXIS_LEFT_Y)
            elif role == 'right_stick_speed':
                cmd = ctrl.stick(AXIS_RIGHT_X)

            if p['invert']:
                cmd = -cmd
            bear.set_goal_velocity((motor_id, cmd * p['limit_velocity_max']))

        now = time.time()
        if now - last_status >= STATUS_INTERVAL_S:
            last_status = now
            parts = []
            for role, motor_id in ROLE_TO_ID.items():
                v, _ = read1(bear.get_present_velocity, motor_id)
                parts.append(f"{role}(id{motor_id}) vel={v:+.2f}rad/s")
            print("\r" + "  ".join(parts) + "   ", end='')
            print(read1(bear.get_present_position, 3))
        tick += 1
        elapsed = time.time() - loop_t0
        time.sleep(max(0.0, dt - elapsed))

except KeyboardInterrupt:
    print("\nInterrupted — stopping.")

finally:
    for motor_id in set(ROLE_TO_ID.values()):
        try:
            bear.set_torque_enable((motor_id, 0))
        except Exception as e:
            print(f"BEAR {motor_id}: disable failed ({e!r}) — pull power if it's still moving.")
    print("\nAll motors disabled.")


Control loop running. Press B to kill, Ctrl-C / stop button to exit safely.

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
left_stick_speed(id3) vel=-0.00rad/s  right_stick_speed(id1) vel=-0.00rad/s  trigger_speed(id2) vel=-0.04rad/s   [PyBEAR | WARNING] :: Single read timed out. Now try again ...
(-2.986509084701538, 128)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
left_stick_speed(id3) vel=+0.00rad/s  right_stick_speed(id1) vel=+0.00rad/s  trigger_speed(id2) vel=-0.01rad/s   [PyBEAR | WARNING] :: Single read timed out. Now try again ...
(-2.9865212440490723, 128)
[PyBEAR | WARN

## EMERGENCY DISABLE

Run any time to cut current. Safe to run repeatedly. **Always targets all three possible IDs
(1, 2, 3)**, not just whatever's currently active in `ROLE_TO_ID` — so if a motor was enabled
earlier and then commented out of Section 1, this still reaches it. IDs with nothing connected
just fail quietly.

In [ ]:
for motor_id in (1, 2, 3):   # intentionally all three, not just ROLE_TO_ID.values() — see markdown above
    try:
        bear.set_torque_enable((motor_id, 0))
        print(f"BEAR {motor_id}: disabled.")
    except Exception as e:
        print(f"BEAR {motor_id}: disable attempt failed ({e!r}). Pull power if still moving.")


## Cleanup

Disable all three possible IDs (same reasoning as EMERGENCY DISABLE) and close the serial port.

In [ ]:
for motor_id in (1, 2, 3):   # intentionally all three, not just ROLE_TO_ID.values()
    try:
        bear.set_torque_enable((motor_id, 0))
    except Exception:
        pass
bear.close_port()
print("Disabled and port closed.")
